In [ ]:
#Cleaning one dataset - load imports
import pandas as pd
!pip install openml
!pip install ucimlrepo

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.3 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=b6f3b5493c36ac0eeac8218c84f05594d172ddde2751f714b30f87d40a3e7cb7
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [ ]:
import numpy as np
import pandas as pd
import cudf

from typing import Optional, Dict, Tuple

from cuml.experimental.preprocessing import IterativeImputer
from cuml.linear_model import BayesianRidge
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler


# -----------------------------
# Basic dataset summaries
# -----------------------------
def profile_target(y: pd.Series) -> Dict:
    """
    Infer task type from the target column.
    """
    y_nonnull = y.dropna()

    if len(y_nonnull) == 0:
        return {"task_type": "unknown", "n_classes": 0}

    if not pd.api.types.is_numeric_dtype(y_nonnull):
        n_classes = y_nonnull.astype(str).nunique()
        if n_classes <= 2:
            return {"task_type": "binary classification", "n_classes": n_classes}
        else:
            return {"task_type": "multiclass classification", "n_classes": n_classes}

    n_unique = y_nonnull.nunique()

    if n_unique <= 10:
        if n_unique <= 2:
            return {"task_type": "binary classification", "n_classes": n_unique}
        else:
            return {"task_type": "multiclass classification", "n_classes": n_unique}

    return {"task_type": "regression", "n_classes": n_unique}


def missingness_summary(df: pd.DataFrame) -> Dict:
    overall_missing_pct = df.isna().mean().mean() * 100
    rows_with_missing_pct = df.isna().any(axis=1).mean() * 100
    return {
        "overall_missing_pct": overall_missing_pct,
        "rows_with_missing_pct": rows_with_missing_pct
    }


def top_missingness_correlations(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    if df.shape[1] > 50:
        return pd.DataFrame(columns=["feature_1", "feature_2", "corr"])

    miss = df.isna().astype(int)

    if miss.shape[1] < 2:
        return pd.DataFrame(columns=["feature_1", "feature_2", "corr"])

    corr = miss.corr()
    pairs = []

    for i in range(len(corr.columns)):
        for j in range(i + 1, len(corr.columns)):
            val = corr.iloc[i, j]
            if pd.notna(val):
                pairs.append((corr.index[i], corr.columns[j], val))

    if not pairs:
        return pd.DataFrame(columns=["feature_1", "feature_2", "corr"])

    out = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "corr"])
    out["abs_corr"] = out["corr"].abs()
    out = out.sort_values("abs_corr", ascending=False).head(top_n).drop(columns="abs_corr")

    return out


# -----------------------------
# Cleaning
# -----------------------------
def clean_dataset(
    df: pd.DataFrame,
    drop_duplicates: bool = True,
    missing_tokens: Optional[list] = None
) -> Tuple[pd.DataFrame, int]:
    df = df.copy()

    if missing_tokens is not None:
        df = df.replace(missing_tokens, np.nan)

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].where(
            df[col].isna(),
            df[col].astype(str).str.strip().str.replace(".", "", regex=False)
        )

    duplicates_dropped = 0
    if drop_duplicates:
        before = len(df)
        df = df.drop_duplicates()
        duplicates_dropped = before - len(df)

    return df, duplicates_dropped


# -----------------------------
# MICE imputation (cuML GPU)
# -----------------------------
def run_mice_imputation(
    X: pd.DataFrame,
    random_state: int = 42,
    max_iter: int = 3
) -> Tuple[pd.DataFrame, IterativeImputer]:
    """
    Runs MICE on numeric dataframe using cuML on GPU.
    Note: cuML BayesianRidge does not support sample_posterior.
    """
    X_gpu = cudf.DataFrame.from_pandas(X)

    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=max_iter,
        random_state=random_state,
        skip_complete=True
    )

    X_imputed_gpu = imputer.fit_transform(X_gpu)
    X_imputed = X_imputed_gpu.to_pandas()
    X_imputed.columns = X.columns
    X_imputed.index = X.index

    return X_imputed, imputer


def artificial_mask_evaluation(
    X: pd.DataFrame,
    random_state: int = 42,
    mask_fraction: float = 0.02,
    max_iter: int = 3
) -> float:
    if X.empty:
        return np.nan

    rng = np.random.default_rng(random_state)
    X_eval = X.copy()

    observed_positions = np.argwhere(~X_eval.isna().values)
    if len(observed_positions) == 0:
        return np.nan

    n_to_mask = int(mask_fraction * len(observed_positions))
    if n_to_mask == 0:
        return np.nan

    chosen_idx = rng.choice(len(observed_positions), size=n_to_mask, replace=False)
    mask_positions = observed_positions[chosen_idx]

    true_values = []
    for r, c in mask_positions:
        true_values.append(X_eval.iat[r, c])
        X_eval.iat[r, c] = np.nan

    X_imputed, _ = run_mice_imputation(
        X_eval,
        random_state=random_state,
        max_iter=max_iter
    )

    pred_values = [X_imputed.iat[r, c] for r, c in mask_positions]

    return mean_squared_error(true_values, pred_values) ** 0.5


# -----------------------------
# Downstream model evaluation
# -----------------------------
def evaluate_downstream_model(
    X: pd.DataFrame,
    y: pd.Series,
    task_type: str,
    random_state: int = 42
) -> Dict:
    keep = ~y.isna()
    X = X.loc[keep].copy()
    y = y.loc[keep].copy()

    if len(X) == 0 or len(y) == 0:
        return {"metric": None, "value": np.nan}

    if not pd.api.types.is_numeric_dtype(y):
        task_type = "binary classification" if y.astype(str).nunique() <= 2 else "multiclass classification"

    stratify_arg = y if task_type != "regression" and y.nunique() > 1 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=stratify_arg
    )

    # Impute train/test on GPU
    X_train_gpu = cudf.DataFrame.from_pandas(X_train)
    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=3,
        random_state=random_state,
        skip_complete=True
    )
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train_gpu).to_pandas().values,
        columns=X_train.columns,
        index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(cudf.DataFrame.from_pandas(X_test)).to_pandas().values,
        columns=X_test.columns,
        index=X_test.index
    )

    if task_type in ["binary classification", "multiclass classification"]:
        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train.astype(str))
        y_test_enc = le.transform(y_test.astype(str))

        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train_imp),
            columns=X_train_imp.columns,
            index=X_train_imp.index
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test_imp),
            columns=X_test_imp.columns,
            index=X_test_imp.index
        )

        model = LogisticRegression(max_iter=2000, random_state=random_state)
        model.fit(X_train_scaled, y_train_enc)
        score = model.score(X_test_scaled, y_test_enc)

        return {"metric": "accuracy", "value": score}

    else:
        y_train_num = pd.to_numeric(y_train, errors="coerce")
        y_test_num = pd.to_numeric(y_test, errors="coerce")

        valid_train = ~y_train_num.isna()
        valid_test = ~y_test_num.isna()

        X_train_imp = X_train_imp.loc[valid_train]
        y_train_num = y_train_num.loc[valid_train]
        X_test_imp = X_test_imp.loc[valid_test]
        y_test_num = y_test_num.loc[valid_test]

        if len(X_train_imp) == 0 or len(X_test_imp) == 0:
            return {"metric": "rmse", "value": np.nan}

        model = Ridge()
        model.fit(X_train_imp, y_train_num)
        preds = model.predict(X_test_imp)
        rmse = mean_squared_error(y_test_num, preds) ** 0.5

        return {"metric": "rmse", "value": rmse}


# -----------------------------
# Main analysis pipeline
# -----------------------------
def analyze_and_impute_dataset(
    df: pd.DataFrame,
    target_col: str,
    dataset_name: str,
    drop_duplicates: bool = True,
    missing_tokens: Optional[list] = None,
    random_state: int = 42,
    run_artificial_mask: bool = False,
    run_downstream: bool = True,
    run_missing_corr: bool = False
) -> Dict:
    df = df.copy()

    df_clean, duplicates_dropped = clean_dataset(
        df, drop_duplicates=drop_duplicates, missing_tokens=missing_tokens
    )

    if target_col not in df_clean.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset '{dataset_name}'")

    y = df_clean[target_col]
    X = df_clean.drop(columns=[target_col])

    X_numeric = X.select_dtypes(include=[np.number]).copy()
    X_numeric = X_numeric.dropna(axis=1, how="all")

    profile = profile_target(y)
    miss_summary = missingness_summary(df_clean)

    missing_corr_pairs = pd.DataFrame()
    if run_missing_corr and not X.empty:
        missing_corr_pairs = top_missingness_correlations(X, top_n=10)

    if not X_numeric.empty:
        X_imputed, _ = run_mice_imputation(X_numeric, random_state=random_state, max_iter=3)
    else:
        X_imputed = X_numeric.copy()

    if run_artificial_mask and not X_numeric.empty:
        imputation_rmse = artificial_mask_evaluation(
            X_numeric, random_state=random_state, mask_fraction=0.02, max_iter=3
        )
    else:
        imputation_rmse = np.nan

    if run_downstream and not X_numeric.empty:
        downstream = evaluate_downstream_model(
            X=X_numeric, y=y, task_type=profile["task_type"], random_state=random_state
        )
        downstream_metric = downstream["metric"]
        downstream_value = downstream["value"]
    else:
        downstream_metric = None
        downstream_value = np.nan

    return {
        "dataset_name": dataset_name,
        "rows_after_cleaning": len(df_clean),
        "duplicates_dropped": duplicates_dropped,
        "task_type": profile["task_type"],
        "n_features": X_numeric.shape[1],
        "overall_missing_pct": miss_summary["overall_missing_pct"],
        "rows_with_missing_pct": miss_summary["rows_with_missing_pct"],
        "imputation_rmse_artificial_mask": imputation_rmse,
        "downstream_metric": downstream_metric,
        "downstream_value": downstream_value,
        "missingness_correlations": missing_corr_pairs,
        "imputed_preview": X_imputed.head(),
        "cleaned_df": df_clean
    }

In [ ]:
from ucimlrepo import fetch_ucirepo
import openml

#Dataset 1
dataset = openml.datasets.get_dataset(45551)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_atlas = pd.concat([X, y], axis=1)
#Dataset 2
dataset = openml.datasets.get_dataset(46888)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_sepsis = pd.concat([X, y], axis=1)
#Dataset 3
dataset = openml.datasets.get_dataset(46860)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_support = pd.concat([X, y], axis=1)
#Dataset 4
dataset = openml.datasets.get_dataset(46882)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_jigsaw = pd.concat([X, y], axis=1)
#Dataset 5
dataset = openml.datasets.get_dataset(46359)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_fraud = pd.concat([X, y], axis=1)
#Dataset 6
dataset = openml.datasets.get_dataset(41147)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_albert = pd.concat([X, y], axis=1)
#Dataset 7
dataset = openml.datasets.get_dataset(45553)
X, y, categorical_indicator, attribute_names = dataset.get_data(target=dataset.default_target_attribute)
df_fico = pd.concat([X, y], axis=1)


datasets = [
    {
        "name": "Atlas Higgs Boson",
        "df": df_atlas,
        "target": "Label",
        "missing_tokens": [-999]
    },
    {
        "name": "Sepsis",
        "df": df_sepsis,
        "target": "SepsisLabel"
    },
    {
        "name": "Support",
        "df": df_support,
        "target": "death"
    },
    {
        "name": "Jigsaw Unintended Bias",
        "df": df_jigsaw,
        "target": "target"
    },
    {
        "name": "Fraud Detection",
        "df": df_fraud,
        "target": "bad_flag"
    },
    {
        "name": "Albert",
        "df": df_albert,
        "target": "class"
    },
    {
        "name": "FICO HELOC",
        "df": df_fico,
        "target": "RiskPerformance", # Added a comma here
        "missing_tokens": [-9, -8, -7]
    }

]

In [ ]:
all_results = []

for d in datasets:
    print(f"Running: {d['name']} | target={d['target']} | shape={d['df'].shape}")
    try:
        res = analyze_and_impute_dataset(
            df=d["df"],
            target_col=d["target"],
            dataset_name=d["name"],
            drop_duplicates=True,
            missing_tokens=d.get("missing_tokens"),
            random_state=42,
            run_artificial_mask=False,
            run_downstream=True,
            run_missing_corr=False
        )
        all_results.append(res)
        print(f"Finished: {d['name']}\n")
    except Exception as e:
        print(f"Failed on {d['name']}: {e}\n")


Running: Atlas Higgs Boson | target=Label | shape=(800000, 31)
Finished: Atlas Higgs Boson

Running: Sepsis | target=SepsisLabel | shape=(1552210, 42)
Finished: Sepsis

Running: Support | target=death | shape=(9105, 47)
Finished: Support

Running: Jigsaw Unintended Bias | target=target | shape=(100000, 44)
Finished: Jigsaw Unintended Bias

Running: Fraud Detection | target=bad_flag | shape=(4156, 29)
Finished: Fraud Detection

Running: Albert | target=class | shape=(425240, 79)
Finished: Albert

Running: FICO HELOC | target=RiskPerformance | shape=(9871, 24)
Finished: FICO HELOC



In [ ]:
results_df = pd.DataFrame([
    {
        "dataset_name": r["dataset_name"],
        "rows_after_cleaning": r["rows_after_cleaning"],
        "duplicates_dropped": r["duplicates_dropped"],
        "task_type": r["task_type"],
        "n_features": r["n_features"],
        "overall_missing_pct": r["overall_missing_pct"],
        "rows_with_missing_pct": r["rows_with_missing_pct"],
        "imputation_rmse_artificial_mask": r["imputation_rmse_artificial_mask"],
        "downstream_metric": r["downstream_metric"],
        "downstream_value": r["downstream_value"]
    }
    for r in all_results
])

results_df


#Change - give me some credit

,dataset_name,rows_after_cleaning,duplicates_dropped,task_type,n_features,overall_missing_pct,rows_with_missing_pct,imputation_rmse_artificial_mask,downstream_metric,downstream_value
0,Atlas Higgs Boson,5000,0,binary classification,30,20.383226,72.74,NaN,accuracy,0.750000
1,Sepsis,5000,0,binary classification,41,66.570952,100.00,NaN,accuracy,0.980000
2,Support,5000,0,binary classification,38,10.964681,96.40,NaN,accuracy,0.854000
3,Jigsaw Unintended Bias,5000,0,binary classification,40,42.934545,86.60,NaN,accuracy,0.978000
4,Fraud Detection,4156,0,binary classification,28,21.335999,100.00,NaN,accuracy,0.899038
5,Albert,5000,0,binary classification,26,8.137468,99.98,NaN,accuracy,0.615000
6,FICO HELOC,5000,0,binary classification,21,5.355833,74.68,NaN,accuracy,0.724000


In [ ]:
import os

output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

for d in datasets:
    print(f"Cleaning full dataset: {d['name']} | original shape={d['df'].shape}")

    df_clean, duplicates_dropped = clean_dataset(
        d["df"],
        drop_duplicates=True,
        missing_tokens=d.get("missing_tokens")
    )

    filename = d["name"].replace(" ", "_").replace("/", "_") + "_FULL_cleaned.csv"
    path = os.path.join(output_dir, filename)

    df_clean.to_csv(path, index=False)

    print(f"Saved: {filename} | cleaned shape={df_clean.shape} | duplicates dropped={duplicates_dropped}")


Cleaning full dataset: Atlas Higgs Boson | original shape=(800000, 31)
Saved: Atlas_Higgs_Boson_FULL_cleaned.csv | cleaned shape=(800000, 31) | duplicates dropped=0
Cleaning full dataset: Sepsis | original shape=(1552210, 42)
Saved: Sepsis_FULL_cleaned.csv | cleaned shape=(1550516, 42) | duplicates dropped=1694
Cleaning full dataset: Support | original shape=(9105, 47)
Saved: Support_FULL_cleaned.csv | cleaned shape=(9105, 47) | duplicates dropped=0
Cleaning full dataset: Jigsaw Unintended Bias | original shape=(100000, 44)
Saved: Jigsaw_Unintended_Bias_FULL_cleaned.csv | cleaned shape=(100000, 44) | duplicates dropped=0
Cleaning full dataset: Fraud Detection | original shape=(4156, 29)
Saved: Fraud_Detection_FULL_cleaned.csv | cleaned shape=(4156, 29) | duplicates dropped=0
Cleaning full dataset: Albert | original shape=(425240, 79)
Saved: Albert_FULL_cleaned.csv | cleaned shape=(425240, 79) | duplicates dropped=0
Cleaning full dataset: FICO HELOC | original shape=(9871, 24)
Saved: FI

In [ ]:
import shutil

shutil.make_archive("./outputs", "zip", "./outputs")
print("Created zip: ./outputs.zip")


Created zip: /content/full_cleaned_datasets.zip


In [ ]:
print("Cleaned datasets saved to ./outputs/")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>